<a href="https://colab.research.google.com/github/Kosala1988/IT5022---Fundamentals-of-Machine-Learning-/blob/main/Used_Car_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fundamentals of Machine Learning: Supervised Learning Project
**Dataset:** Used Car Price Prediction Challenge
**Source:** [Kaggle - Car Price Prediction Challenge](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge/data)

## Phase 1: Context & Setup
**1.1 Context**
The objective of this project is to apply supervised machine learning algorithms to predict the market price of used cars based on various historical and physical attributes. The dataset contains records of cars scraped from an online marketplace, presenting a real-world regression problem with mixed data types (categorical and continuous) and messy formatting that requires rigorous preprocessing.

In [6]:
# ==============================================================================
# Phase 1: Environment Setup & Data Ingestion
# ==============================================================================

# 1.0 Library Imports
# Importing core libraries required for data manipulation, visualization,
# and machine learning model development.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn modules for preprocessing, model selection, and evaluation
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore') # Suppress warnings for clean report output

In [7]:
# 1.1 Data Ingestion
# Loading the raw Used Car Price Prediction dataset into a Pandas DataFrame.
# Source: Kaggle (Deep Contractor)
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/FML Assignment/car_price_prediction.csv')

In [8]:
print("First 5 records:", df.head())

First 5 records:          ID  Price  Levy Manufacturer    Model  Prod. year   Category  \
0  45654403  13328  1399        LEXUS   RX 450        2010       Jeep   
1  44731507  16621  1018    CHEVROLET  Equinox        2011       Jeep   
2  45774419   8467     -        HONDA      FIT        2006  Hatchback   
3  45769185   3607   862         FORD   Escape        2011       Jeep   
4  45809263  11726   446        HONDA      FIT        2014  Hatchback   

  Leather interior Fuel type Engine volume    Mileage  Cylinders  \
0              Yes    Hybrid           3.5  186005 km        6.0   
1               No    Petrol             3  192000 km        6.0   
2               No    Petrol           1.3  200000 km        4.0   
3              Yes    Hybrid           2.5  168966 km        4.0   
4              Yes    Petrol           1.3   91901 km        4.0   

  Gear box type Drive wheels   Doors             Wheel   Color  Airbags  
0     Automatic          4x4  04-May        Left wheel  Silve

**1.2 Dataset Attributes (Parameters)**
The dataset consists of the following key features:
* **Target Variable:** `Price` (Continuous) - The listing price of the vehicle.
* **Continuous Features:** `Prod. year`, `Mileage` (distance traveled), `Engine volume` (capacity), `Cylinders`.
* **Categorical Features:** `Manufacturer`, `Model`, `Category` (e.g., Jeep, Hatchback), `Leather interior`, `Fuel type`, `Gear box type`, `Drive wheels`, `Doors`, `Wheel` (Left/Right hand drive), `Color`.
* **Administrative/Other:** `ID` (Unique identifier), `Levy` (Tax/Import fee, contains missing values).

In [9]:
# 1.3 Dataset Size and Structure Checks
print("--- Dataset Size ---")
rows, columns = df.shape
print(f"Total Rows (Records): {rows}")
print(f"Total Columns (Features): {columns}\n")

print("--- Data Types and Non-Null Counts ---")
# info() is great, but we capture it cleanly for our output
df.info()

print("\n--- Summary Statistics (Continuous Variables) ---")
# Transpose (.T) makes it much easier to read when you have many columns
display(df.describe().T)

print("\n--- Summary Statistics (Categorical Variables) ---")
display(df.describe(include=['object']).T)

--- Dataset Size ---
Total Rows (Records): 19237
Total Columns (Features): 18

--- Data Types and Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors           

,count,mean,std,min,25%,50%,75%,max
ID,19237.0,4.557654e+07,936591.422799,20746880.0,45698374.0,45772308.0,45802036.0,45816654.0
Price,19237.0,1.855593e+04,190581.269684,1.0,5331.0,13172.0,22075.0,26307500.0
Prod. year,19237.0,2.010913e+03,5.668673,1939.0,2009.0,2012.0,2015.0,2020.0
Cylinders,19237.0,4.582991e+00,1.199933,1.0,4.0,4.0,4.0,16.0
Airbags,19237.0,6.582627e+00,4.320168,0.0,4.0,6.0,12.0,16.0



--- Summary Statistics (Categorical Variables) ---


,count,unique,top,freq
Levy,19237,559,-,5819
Manufacturer,19237,65,HYUNDAI,3769
Model,19237,1590,Prius,1083
Category,19237,11,Sedan,8736
Leather interior,19237,2,Yes,13954
Fuel type,19237,7,Petrol,10150
Engine volume,19237,107,2,3916
Mileage,19237,7687,0 km,721
Gear box type,19237,4,Automatic,13514
Drive wheels,19237,3,Front,12874


In [10]:
# check Missing value
df.isnull().sum()

,0
ID,0
Price,0
Levy,0
Manufacturer,0
Model,0
Prod. year,0
Category,0
Leather interior,0
Fuel type,0
Engine volume,0


In [11]:
# Check Duplication
df.duplicated().sum()

np.int64(313)

In [12]:
#Check datatype
df.dtypes

,0
ID,int64
Price,int64
Levy,object
Manufacturer,object
Model,object
Prod. year,int64
Category,object
Leather interior,object
Fuel type,object
Engine volume,object


In [13]:
# Check the number of unique values of each column
df.nunique()

,0
ID,18924
Price,2315
Levy,559
Manufacturer,65
Model,1590
Prod. year,54
Category,11
Leather interior,2
Fuel type,7
Engine volume,107


In [14]:
#Check statistics of data set
desc = ['Price','Levy','Mileage' ,'Prod. year', 'Cylinders', 'Airbags']
df[desc].describe()

,Price,Prod. year,Cylinders,Airbags
count,1.923700e+04,19237.000000,19237.000000,19237.000000
mean,1.855593e+04,2010.912824,4.582991,6.582627
std,1.905813e+05,5.668673,1.199933,4.320168
min,1.000000e+00,1939.000000,1.000000,0.000000
25%,5.331000e+03,2009.000000,4.000000,4.000000
50%,1.317200e+04,2012.000000,4.000000,6.000000
75%,2.207500e+04,2015.000000,4.000000,12.000000
max,2.630750e+07,2020.000000,16.000000,16.000000


In [15]:
# Check unique values for the actual categorical columns in our dataset
categorical_columns = [
    'Manufacturer', 'Category', 'Leather interior',
    'Fuel type', 'Gear box type', 'Drive wheels',
    'Doors', 'Wheel', 'Color'
]

# Note: 'Model' is excluded here because it has 1590 unique values,
# which will flood your screen!

for col in categorical_columns:
    print(f"--- Unique categories in {col} ---")
    print(df[col].unique())
    print("\n")

--- Unique categories in Manufacturer ---
['LEXUS' 'CHEVROLET' 'HONDA' 'FORD' 'HYUNDAI' 'TOYOTA' 'MERCEDES-BENZ'
 'OPEL' 'PORSCHE' 'BMW' 'JEEP' 'VOLKSWAGEN' 'AUDI' 'RENAULT' 'NISSAN'
 'SUBARU' 'DAEWOO' 'KIA' 'MITSUBISHI' 'SSANGYONG' 'MAZDA' 'GMC' 'FIAT'
 'INFINITI' 'ALFA ROMEO' 'SUZUKI' 'ACURA' 'LINCOLN' 'VAZ' 'GAZ' 'CITROEN'
 'LAND ROVER' 'MINI' 'DODGE' 'CHRYSLER' 'JAGUAR' 'ISUZU' 'SKODA'
 'DAIHATSU' 'BUICK' 'TESLA' 'CADILLAC' 'PEUGEOT' 'BENTLEY' 'VOLVO' 'სხვა'
 'HAVAL' 'HUMMER' 'SCION' 'UAZ' 'MERCURY' 'ZAZ' 'ROVER' 'SEAT' 'LANCIA'
 'MOSKVICH' 'MASERATI' 'FERRARI' 'SAAB' 'LAMBORGHINI' 'ROLLS-ROYCE'
 'PONTIAC' 'SATURN' 'ASTON MARTIN' 'GREATWALL']


--- Unique categories in Category ---
['Jeep' 'Hatchback' 'Sedan' 'Microbus' 'Goods wagon' 'Universal' 'Coupe'
 'Minivan' 'Cabriolet' 'Limousine' 'Pickup']


--- Unique categories in Leather interior ---
['Yes' 'No']


--- Unique categories in Fuel type ---
['Hybrid' 'Petrol' 'Diesel' 'CNG' 'Plug-in Hybrid' 'LPG' 'Hydrogen']


--- Unique cat